In [10]:
import pandas as pd

FILE = r"C:\Users\nay\Desktop\qr\qr\utils\510300.XSHG.csv"

df = pd.read_csv(FILE, encoding="utf-8-sig")  # 去掉可能的 BOM
df = df.rename(columns={"Unnamed: 0": "order_book_id", "Unnamed: 1": "date"})
# 如果不需要 order_book_id，可丢弃或设为索引
df = df.drop(columns=["order_book_id"])
# df = df.set_index("order_book_id")

# 如果要把 date 解析为日期类型
df["date"] = pd.to_datetime(
    df["date"].str.strip(),
    format="%m/%d/%Y",  # 如果是两位年份就用 %y
    errors="coerce",
)

print(df.columns.tolist())
print(df.head())


['date', 'prev_close', 'volume', 'iopv', 'low', 'high', 'limit_down', 'close', 'num_trades', 'limit_up', 'open', 'total_turnover']
        date  prev_close       volume    iopv    low   high  limit_down  \
0        NaT         NaN          NaN     NaN    NaN    NaN         NaN   
1 2025-12-01       4.635  578301370.0  4.6887  4.639  4.688       4.172   
2 2025-12-02       4.687  383444900.0  4.6691  4.651  4.686       4.218   
3 2025-12-03       4.666  395034780.0  4.6425  4.635  4.684       4.199   
4 2025-12-04       4.645  402618836.0  4.6578  4.628  4.671       4.181   

   close  num_trades  limit_up   open  total_turnover  
0    NaN         NaN       NaN    NaN             NaN  
1  4.687     57000.0     5.099  4.646    2.698559e+09  
2  4.666     41246.0     5.156  4.686    1.790232e+09  
3  4.645     51971.0     5.133  4.665    1.840202e+09  
4  4.657     42547.0     5.110  4.648    1.871832e+09  


In [12]:
# Mean Reversion: Betting that if it falls too much, it will rise, and if it rises too much, it will fall (catching a falling knife).
'''
Indicator Calculation:
    Middle Band: Average closing price over the past 20 days.
    Standard Deviation: Standard deviation of closing prices over the past 20 days.
    Upper Band: Middle Band + 2 standard deviations.
    Lower Band: Middle Band - 2 standard deviations.

Trading Signals:
    Buy (Entry): When yesterday's closing price < yesterday's lower band. (The price has fallen too much; buy the dip.)
    Sell/Exit (Exit): When yesterday's closing price > yesterday's upper band. (The price has risen too much; exit.)
    Hold: If in the middle range, maintain the previous position.
'''

import numpy as np
import pandas as pd

np.random.seed(42) # 保证每次生成的随机数一样，方便我检查
days = 200
# 模拟一个从100块开始，每天波动在-3%到+3%之间的股票
price_changes = np.random.normal(0, 0.03, days) 
price = 100 * (1 + price_changes).cumprod()
df = pd.DataFrame({
    'price':price
})
df['incOrNot'] = df['price'].pct_change()
df['mean20'] = df['price'].rolling(window=20).mean()
df['std20'] = df['price'].rolling(window=20).std()
df['upper'] = df['mean20'] + 2 * df['std20']
df['lower'] = df['mean20'] - 2 * df['std20']
# 1 buy 0 sell 
entry_event  = (df['price'] < (df['lower'])).fillna(False).astype(bool)
exit_event = (df['price'] > (df['upper'])).fillna(False).astype(bool)

pos = pd.Series(np.nan,index=df.index)
pos[entry_event] = 1
pos[exit_event] = 0
df['position'] = pos.ffill().fillna(0).astype(int)

df['strategyGains'] = (1+(df['incOrNot'] *df['position'].shift(1).fillna(0))).cumprod()-1  
df['brainlessFixedInvestment'] = (1+df['incOrNot']).cumprod()-1
print(df)

          price  incOrNot     mean20     std20      upper      lower  \
0    101.490142       NaN        NaN       NaN        NaN        NaN   
1    101.069169 -0.004148        NaN       NaN        NaN        NaN   
2    103.033009  0.019431        NaN       NaN        NaN        NaN   
3    107.740679  0.045691        NaN       NaN        NaN        NaN   
4    106.983844 -0.007025        NaN       NaN        NaN        NaN   
..          ...       ...        ...       ...        ...        ...   
195   76.562352  0.011560  78.910817  3.013139  84.937095  72.884539   
196   74.532246 -0.026516  78.953611  2.940076  84.833763  73.073459   
197   74.875970  0.004612  78.852954  3.046993  84.946941  72.758967   
198   75.006723  0.001746  78.789359  3.116042  85.021443  72.557275   
199   72.434810 -0.034289  78.285933  3.292755  84.871443  71.700423   

     position  strategyGains  brainlessFixedInvestment  
0           0            NaN                       NaN  
1           0        

In [2]:
import numpy as np
import pandas as pd

price = [100, 102, 99, 97, 101, 105, 103, 108, 106, 110, 115, 112, 118, 120, 117]
day = [i for i in range(1, len(price)+1)]
df = pd.DataFrame({
    'day': day,
    'price': price
})

# 1. 计算涨跌幅
df['incOrNot'] = df['price'].pct_change()

# 2. 计算均线
df['AverageThreeDays'] = df['price'].rolling(window=3).mean()
df['AverageFiveDays'] = df['price'].rolling(window=5).mean()

# 3. 捕捉金叉/死叉 (你的逻辑完全正确)
# 这里的 * 等同于 "AND"
golden_cross = (df['AverageThreeDays'] > df['AverageFiveDays']) & (
    df['AverageThreeDays'].shift(1) < df['AverageFiveDays'].shift(1)
)
death_cross = (df['AverageThreeDays'] < df['AverageFiveDays']) & (
    df['AverageThreeDays'].shift(1) > df['AverageFiveDays'].shift(1)
)

# 4. 生成信号 (State)
df['signalBuyOrNot'] = np.nan # 初始化为 NaN
df.loc[golden_cross, 'signalBuyOrNot'] = 1 # 金叉天标记 1
df.loc[death_cross, 'signalBuyOrNot'] = 0  # 死叉天标记 0

# 核心：向前填充 (ffill)
# 只要没遇到新的死叉，就一直保持 1；只要没遇到新金叉，就一直保持 0
df['signalBuyOrNot'] = df['signalBuyOrNot'].ffill()

# 填补最开始的空缺：在第一个信号出现之前，默认空仓 (0)
df['signalBuyOrNot'] = df['signalBuyOrNot'].fillna(0)

# 5. 计算收益 (核心修正！！！)
# 昨天收盘的信号，决定今天的收益
# 昨天的信号是 signal.shift(1)
# 今天的涨跌是 incOrNot
df['strategy_daily_return'] = df['incOrNot'] * df['signalBuyOrNot'].shift(1)

# 净值计算：1 + 策略收益，然后累乘
df['strategyGains'] = (1 + df['strategy_daily_return']).cumprod() 

# 6. 对比基准
df['brainlessFixedInvestment'] = (1 + df['incOrNot']).cumprod()

# 打印关键列
print(df[['day', 'price', 'AverageThreeDays', 'AverageFiveDays', 'signalBuyOrNot', 'strategyGains']])

    day  price  AverageThreeDays  AverageFiveDays  signalBuyOrNot  \
0     1    100               NaN              NaN             0.0   
1     2    102               NaN              NaN             0.0   
2     3     99        100.333333              NaN             0.0   
3     4     97         99.333333              NaN             0.0   
4     5    101         99.000000             99.8             0.0   
5     6    105        101.000000            100.8             1.0   
6     7    103        103.000000            101.0             1.0   
7     8    108        105.333333            102.8             1.0   
8     9    106        105.666667            104.6             1.0   
9    10    110        108.000000            106.4             1.0   
10   11    115        110.333333            108.4             1.0   
11   12    112        112.333333            110.2             1.0   
12   13    118        115.000000            112.2             1.0   
13   14    120        116.666667  

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
                   "A": [np.nan, 1, np.nan, np.nan, 2],
                   "B": [2, np.nan, np.nan, np.nan, 4],
                   "C": [np.nan, 3, np.nan, np.nan, 5],
                   "D": [4, 5, np.nan, 8, 9]
                  })

# 后填操作
# print(df.fillna(0)) 
print(df.bfill()) 

# 前填操作
print(df.ffill())


     A    B    C    D
0  1.0  2.0  3.0  4.0
1  1.0  4.0  3.0  5.0
2  2.0  4.0  5.0  8.0
3  2.0  4.0  5.0  8.0
4  2.0  4.0  5.0  9.0
     A    B    C    D
0  NaN  2.0  NaN  4.0
1  1.0  2.0  3.0  5.0
2  1.0  2.0  3.0  5.0
3  1.0  2.0  3.0  8.0
4  2.0  4.0  5.0  9.0


In [8]:
# Mean Reversion: Betting that if it falls too much, it will rise, and if it rises too much, it will fall (catching a falling knife).
'''
Indicator Calculation:
    Middle Band: Average closing price over the past 20 days.
    Standard Deviation: Standard deviation of closing prices over the past 20 days.
    Upper Band: Middle Band + 2 standard deviations.
    Lower Band: Middle Band - 2 standard deviations.

Trading Signals:
    Buy (Entry): When yesterday's closing price < yesterday's lower band. (The price has fallen too much; buy the dip.)
    Sell/Exit (Exit): When yesterday's closing price > yesterday's upper band. (The price has risen too much; exit.)
    Hold: If in the middle range, maintain the previous position.
'''

import numpy as np
import pandas as pd

np.random.seed(42) # 保证每次生成的随机数一样，方便我检查
days = 200
# 模拟一个从100块开始，每天波动在-3%到+3%之间的股票
price_changes = np.random.normal(0, 0.03, days) 
price = 100 * (1 + price_changes).cumprod()


df = pd.DataFrame({
    'price':price
})
df['mean20'] = df['price'].rolling(window=20).mean()
df['std20'] = df['price'].rolling(window=20).std()
df['upper'] = df['mean20'] + 2 * df['std20']
df['lower'] = df['mean20'] - 2 * df['std20']
# 1 buy 2 sell 0 remain
buySignal = (df['price'] < (df['lower'])).shift(1)
sellSignal = (df['price'] > (df['upper'])).shift(1)
buySignal[0] = buySignal[1]
sellSignal[0] = sellSignal[1]
df['signalBuyOrNot'] = np.nan
df.loc[buySignal,'signalBuyOrNot'] = 1
df.loc[sellSignal,'signalBuyOrNot'] = 2
df['signalBuyOrNot'] = df['signalBuyOrNot'].ffill()
print(df['signalBuyOrNot'])

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
195    2.0
196    2.0
197    2.0
198    2.0
199    2.0
Name: signalBuyOrNot, Length: 200, dtype: float64


In [7]:
import numpy as np
import pandas as pd

price = [100, 102, 99, 97, 101, 105, 103, 108, 106, 110, 115, 112, 118, 120, 117]
day = [i for i in range(1,len(price)+1)]
df = pd.DataFrame({
    'day':day,
    'price':price
})

df['incOrNot'] = df['price'].pct_change()

df['AverageThreeDays'] = df['price'].rolling(window=3).mean()
df['AverageFiveDays'] = df['price'].rolling(window=5).mean()

golden_cross = (df['AverageThreeDays'] > df['AverageFiveDays'])*(
    df['AverageThreeDays'].shift(1) < df['AverageFiveDays'].shift(1)
)
death_cross = (df['AverageThreeDays'] < df['AverageFiveDays'])*(
    df['AverageThreeDays'].shift(1) > df['AverageFiveDays'].shift(1)
)

df['signalBuyOrNot'] = np.nan
df.loc[golden_cross,'signalBuyOrNot'] = 1
print(df)
df.loc[death_cross,'signalBuyOrNot'] = 0
print(df)
df['signalBuyOrNot'] = df['signalBuyOrNot'].ffill()
print(df)

    day  price  incOrNot  AverageThreeDays  AverageFiveDays  signalBuyOrNot
0     1    100       NaN               NaN              NaN             NaN
1     2    102  0.020000               NaN              NaN             NaN
2     3     99 -0.029412        100.333333              NaN             NaN
3     4     97 -0.020202         99.333333              NaN             NaN
4     5    101  0.041237         99.000000             99.8             NaN
5     6    105  0.039604        101.000000            100.8             1.0
6     7    103 -0.019048        103.000000            101.0             NaN
7     8    108  0.048544        105.333333            102.8             NaN
8     9    106 -0.018519        105.666667            104.6             NaN
9    10    110  0.037736        108.000000            106.4             NaN
10   11    115  0.045455        110.333333            108.4             NaN
11   12    112 -0.026087        112.333333            110.2             NaN
12   13    1